# 37. Read the plasticity indicators

Notebook 36 writes one JSON line per task boundary, one file per run. This
notebook reads those files on a CPU runtime and turns them into three things:

1. **A long table.** One row per measurement, with the same columns as
   `study_long.csv` in `reference/adapter-plasticity-study`, so the two sets of
   numbers can be put side by side without reshaping either one.
2. **Curves.** Each measure against task position, one line per run, with the
   three fixed image sets in separate panels.
3. **An endpoint table.** Mean and spread across runs at the first and last
   boundary, and the change between them.

Everything after the long table is computed from the long table, never from
the records again. A figure that disagrees with the table is then a bug in one
line of code rather than a second measurement to chase.

### The question these answer

Does the generator show the wear the adapter plasticity study found in
adapters? That study reports indicators that move a great deal over a stream.
A flat line here is a result, not a failure.

### What is deliberately missing

The adapter study's headline figure is a scatter of probe accuracy against each
indicator. It probes at four points; we probe at two. Three seeds would give
six points, which is too few to say whether an indicator tracks the probe. The
curve and the endpoint table are what two probe points honestly support.

This notebook downloads records, never models. Do not occupy a GPU for it.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.indicator_report import (read_rows, long_table, write_long_table,
                                    endpoints, curves)

import torch
if torch.cuda.is_available():
    raise RuntimeError("Switch to a CPU runtime. This notebook reads records, not models.")
print("Reading at:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

## 2. Choose the study

In [ ]:
STUDY_ID = "indicator_stream_v1"   # the study notebook 36 wrote to
BUCKET = "uncle-experiments"
SAVE_REPORT_TO_R2 = True
print("Study:", STUDY_ID)

## 3. Download every run's record

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/indicators" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)

# One .jsonl per run, under order and seed. The .baseline.pt files beside them
# belong to the runs, not to the reading, so they are skipped here.
PREFIX = f"uncle/indicators/{STUDY_ID}/"
found = sorted(key for key in store.keys(PREFIX) if key.endswith(".jsonl"))
if not found:
    raise RuntimeError(f"No indicator records under {PREFIX}. Run notebook 36 first.")

paths = [store.download(key, CACHE / "cache") for key in found]
for key in found:
    print("  ", key)
print(f"\n{len(paths)} run files.")

## 4. Check what each run holds

In [ ]:
rows = read_rows(*paths)
runs = {}
for row in rows:
    runs.setdefault((row["order"], row["seed"]), []).append(row)

print(f"{'run':<22} {'boundaries':>10}  tasks learned")
for (order, seed), lines in sorted(runs.items()):
    learned = [line["task"] for line in sorted(lines, key=lambda r: r["t"]) if line["task"]]
    print(f"{order + '/seed_' + str(seed):<22} {len(lines):>10}  {' '.join(learned) or 'none yet'}")

# A run is finished when it has boundary zero plus one line per task. A shorter
# run is still readable; it just stops earlier on every curve, and the endpoint
# table below compares each measure at the first and last boundary present
# across all runs, so an unfinished run shortens the comparison for everyone.
lengths = {len(lines) for lines in runs.values()}
if len(lengths) > 1:
    print("\nThese runs have different numbers of boundaries:", sorted(lengths))
    print("Read the endpoint table with that in mind, or rerun the short ones to the end.")

## 5. The long table

One row per measurement. Lines that name the run or the boundary are dropped,
and so is anything that is not a single number: the loss progress curve is a
list, and the loss progress mean is empty at boundary zero, where no task was
learned yet.

In [ ]:
flat = long_table(rows)
TABLE = CACHE / f"indicator_long_{STUDY_ID}.csv"
write_long_table(flat, TABLE)
print(len(flat), "rows written to", TABLE.name)

# How many images went through and how many units were counted. They are in
# the record so a reader can check that every boundary measured the same thing,
# and they are constant by construction, so they are kept out of the tables and
# figures below rather than drawn as flat lines.
CONSTANTS = ("images", "units")
for measure in CONSTANTS:
    seen = {row["value"] for row in flat if row["measure"] == measure}
    print(f"{measure}: {sorted(seen)}"
          + ("" if len(seen) == 1 else "   <- not constant, check the runs"))

moving = [row for row in flat if row["measure"] not in CONSTANTS]
measures = sorted({row["measure"] for row in moving})
sets = sorted({row["image_set"] for row in moving})
print("\nMeasures:", ", ".join(measures))
print("Image sets:", ", ".join(name or "(none)" for name in sets))
print("\nFirst few rows:")
print(f"{'run':<18} {'t':>3} {'image set':<12} {'measure':<22} {'value':>10}")
for row in moving[:12]:
    print(f"{row['run']:<18} {row['t']:>3} {row['image_set'] or '-':<12} "
          f"{row['measure']:<22} {row['value']:>10.4f}")

## 6. What moved, and by how much

The spread is the sample standard deviation across runs, the same figure the
probe tables report, so an indicator's change can be read next to the probe's
change in the same units of evidence.

A change is worth attention when it is larger than the spread it sits in. The
table lays the two numbers side by side; it does not make that call.

In [ ]:
summary = endpoints(moving)

# The two boundary columns are not decoration. Loss progress has no value at
# boundary zero, where no task had been learned, so its row compares a later
# pair of boundaries than the rest and would otherwise look like the same
# comparison.
print(f"{'measure':<22} {'image set':<12} {'runs':>5} {'t0':>3} {'t1':>3} "
      f"{'start':>10} {'sd':>8} {'end':>10} {'sd':>8} {'change':>10}")
print("-" * 100)
for row in summary:
    change = "" if row["change"] is None else f"{row['change']:.4f}"
    print(f"{row['measure']:<22} {row['image_set'] or '-':<12} {row['runs']:>5} "
          f"{row['first_t']:>3} {row['last_t']:>3} "
          f"{row['start_mean']:>10.4f} {row['start_sd']:>8.4f} "
          f"{row['end_mean']:>10.4f} {row['end_sd']:>8.4f} {change:>10}")

print("\nThe spread is across runs, not across tasks. One run gives no spread.")

## 7. The curves

One figure per measure. Panels split the image sets, because a measure that
moves on the stream's own images and stays flat on a different dataset is
saying something different from one that moves on both. That split is the
correction the adapter study had to make after the fact, so it is built in
here.

In [ ]:
FIGURES = CACHE / "figures"
written = curves(moving, FIGURES)
print(len(written), "figures in", FIGURES)

from IPython.display import Image, display
for path in written:
    print("\n" + path.stem.replace("fig_", ""))
    display(Image(filename=str(path)))

## 8. Save the reading

The table, the figures and a short record of what was read go to R2 beside the
runs, so the reading can be checked later against the exact files it used.

In [ ]:
import datetime, platform

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
record = {
    "notebook": "37_indicator_report",
    "study_id": STUDY_ID,
    "read_at": stamp,
    "source_keys": found,
    "runs": [{"order": order, "seed": seed, "boundaries": len(lines)}
             for (order, seed), lines in sorted(runs.items())],
    "measures": measures,
    "image_sets": [name for name in sets if name],
    "endpoints": summary,
    "table": ("The CSV holds every measurement, including the image and unit "
              "counts that the tables and figures leave out."),
    "limitations": [
        "The spread is across runs at the same settings, so it measures seed "
        "variation and nothing else.",
        "Rank and dormancy are read on fixed image sets through one fixed "
        "reference code, so they describe the network the generator writes, "
        "not the network any one task uses.",
        "Loss progress is sampled every tenth update, and its mean covers one "
        "task, so a task with fewer updates contributes fewer samples.",
        "No probe accuracy is joined to these numbers. Two probe points cannot "
        "say whether an indicator tracks the probe.",
    ],
    "environment": {"python": platform.python_version(), "torch": torch.__version__},
}

REPORT = CACHE / f"indicator_report_{STUDY_ID}_{stamp}"
REPORT.mkdir(parents=True, exist_ok=True)
(REPORT / "reading.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
(REPORT / TABLE.name).write_bytes(TABLE.read_bytes())
for path in written:
    (REPORT / path.name).write_bytes(path.read_bytes())
print("Saved locally to", REPORT)

if SAVE_REPORT_TO_R2:
    base = f"uncle/indicator_reports/{STUDY_ID}/{stamp}"
    for path in sorted(REPORT.iterdir()):
        store.upload(path, f"{base}/{path.name}", overwrite=True)
    print("Uploaded to", base)
else:
    print("Not uploaded. Set SAVE_REPORT_TO_R2 to save this reading in R2.")

## 9. Reading the result

Take the endpoint table first. For each measure, compare the change against the
spread on the same row. A change smaller than the spread is not yet a finding,
whatever the curve looks like.

Then look at which image sets moved. A measure that falls on the stream's own
images while holding steady on EuroSAT is about the data the model was trained
on, not about the network's capacity. One that falls on all three is about the
network.

Read the three weight measures together, never one alone. `delta_w` says how
far the weights travelled, `norm` says whether they grew, and `angle` says
whether they turned. Weights that only grow travel a long way without pointing
anywhere new, so a large `delta_w` beside a near zero `angle` is a different
story from a large `delta_w` beside a large one, even though the first number
is the same in both.

The record in `docs/wiki/Experiment-trajectory.md` takes the numbers from the
endpoint table and the file names from section 8, so the reading can be traced
back to the exact run files.